# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aisyahnabillah/ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule, in plain words: a page is worth reviewing **if it hasn't been updated in 180+ days (stale) and it still gets meaningful traffic (visible, 500+ impressions in the last 90 days).** Among pages meeting both conditions, **rank by how much traffic they still get**, since that's where review effort pays off most.

Reason codes this rule can output:
- `stale_but_visible`: page is stale and visible, gets flagged for review
- `not_flagged`: page doesn't meet both conditions, no action needed right now

**Signal 1** (staleness, behind FlyRank's refresh flags) and **Signal 2** (CTR vs position, behind CTR-fix logic) are checked below with bucket tables before I trust them in the rule.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)

bucket1 = df.groupby("stale").agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
).reset_index()
print(bucket1)

   stale      n  decline_rate
0      0  29826      0.542480
1      1    174      0.471264


Signal 1 verdict: **OPPOSITE** (small sample, interpret cautiously)

Reasoning: stale pages (n=174) actually show a LOWER decline rate (47.1%) than non-stale pages (n=29,826, decline rate 54.2%), the opposite of what "staleness predicts decline" would suggest. However, the stale group is a very small slice of the data, so this direction should be trusted with caution, not treated as a strong finding. This doesn't invalidate the `stale_but_visible` reason code as a review trigger, but it does mean "declining" is not the right justification for why staleness matters, that assumption doesn't hold here.

In [3]:
df["position_tier"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 100],
    labels=["top_3", "page_1", "page_2", "beyond"]
)

bucket2 = df.groupby("position_tier", observed=True).agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
).reset_index()
print(bucket2)

  position_tier      n  mean_ctr
0         top_3   1141  2.714303
1        page_1  11842  0.651045
2        page_2   7273  0.323443
3        beyond   8524  0.211705


Signal 2 verdict: **CONFIRMED**

Reasoning: mean CTR drops consistently from top_3 (2.71) to page_1 (0.65) to page_2 (0.32) to beyond (0.21), a clean monotonic pattern across a well-sized bucket in each tier (n ranges from 1,141 to 11,842). This supports using position as a meaningful signal, consistent with the weak-but-real pattern from the overall correlation check in w01.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = stale × visible × impressions_90d. 

Deliberately simple, readable, no fitted weights,  just conditions multiplied together, per the baseline skill's rule.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

df["baseline_action_score"] = df["stale"] * df["visible"] * df["impressions_90d"]

df["reason_code"] = np.where(
    (df["stale"] == 1) & (df["visible"] == 1),
    "stale_but_visible",
    "not_flagged"
)

df["action_label"] = np.where(
    df["baseline_action_score"] > 0,
    "review_for_refresh",
    "monitor"
)

queue = df.sort_values("baseline_action_score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

queue[["content_id", "baseline_action_score", "reason_code", "action_label"]].head(10)

,content_id,baseline_action_score,reason_code,action_label
16751,content_cf56e2e2e282,61678,stale_but_visible,review_for_refresh
16514,content_7368877ea310,59472,stale_but_visible,review_for_refresh
7021,content_1bfaa38ff26c,25715,stale_but_visible,review_for_refresh
21268,content_0a91db491d14,13299,stale_but_visible,review_for_refresh
11489,content_5feee3994adb,7812,stale_but_visible,review_for_refresh
12045,content_c2d929d83eaa,7558,stale_but_visible,review_for_refresh
698,content_b16bd7307b39,4590,stale_but_visible,review_for_refresh
5327,content_fe16a55cd13d,4556,stale_but_visible,review_for_refresh
26810,content_ecb6215e79fd,4429,stale_but_visible,review_for_refresh
20837,content_928af3e22c80,1697,stale_but_visible,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
top20[["content_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction", "action_label", "reason_code"]]

,content_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,action_label,reason_code
16751,content_cf56e2e2e282,61678,194,19.7,0.15,down,review_for_refresh,stale_but_visible
16514,content_7368877ea310,59472,194,24.8,0.13,down,review_for_refresh,stale_but_visible
7021,content_1bfaa38ff26c,25715,194,22.2,0.23,down,review_for_refresh,stale_but_visible
21268,content_0a91db491d14,13299,193,10.5,0.49,down,review_for_refresh,stale_but_visible
11489,content_5feee3994adb,7812,194,39.0,0.01,down,review_for_refresh,stale_but_visible
12045,content_c2d929d83eaa,7558,193,17.9,0.20,down,review_for_refresh,stale_but_visible
698,content_b16bd7307b39,4590,194,31.0,0.00,down,review_for_refresh,stale_but_visible
5327,content_fe16a55cd13d,4556,194,16.4,0.33,down,review_for_refresh,stale_but_visible
26810,content_ecb6215e79fd,4429,194,25.3,0.38,down,review_for_refresh,stale_but_visible
20837,content_928af3e22c80,1697,193,15.8,0.12,down,review_for_refresh,stale_but_visible


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick:** rows 18-20 in this top-20 (content_c87291853cab, content_3dc420aa9809, content_6f2f3043b633) are not actually flagged by the rule, action_label = "monitor" and reason_code = "not_flagged", meaning baseline_action_score = 0. They only appear in this list because the entire dataset has just 17 pages with a nonzero score (only 
174 pages are even "stale", and fewer than that also clear the visibility bar), so these three are tied at zero and landed here by sort order, not because the rule judged them worth reviewing. A true "top-20" for this rule really only has 17 meaningful entries.

**Leakage check:** the score only uses days_since_last_update and impressions_90d, both knowable before any outcome is known. trend_direction and ctr were used only to verify Signal 1 and Signal 2, never as scoring inputs. No product-decision flags exist in this  dataset to leak from.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Leakage check (confirm no product flags or future-window inputs used)
print(top20.columns.tolist())
print("trend_direction used only for signal verification, not as a scoring input:", 
      "trend_direction" not in ["stale", "visible", "baseline_action_score"])

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'stale', 'visible', 'baseline_action_score', 'reason_code', 'action_label']
trend_direction used only for signal verification, not as a scoring input: True


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.